# The authors' released models, scored on RelBench's test labels

Runs the authors' code ([cy623/Structural_Optimizer_RDL](https://github.com/cy623/Structural_Optimizer_RDL), commit `030b07c`) **unmodified**, in its own environment (their `install.sh`: torch 2.7.1, PyG 2.6.1, PyTorch Frame 0.2.5), with their released weights. For each task: `main.py --mode prepare`, `main.py --mode predict`, then `experiments/benchmarks/score_authors_predictions.py` (ours) computes test ROC-AUC against RelBench's test labels and lists the paper's number.

Their release covers 9 of their 12 tasks (no weights for rel-trial study-outcome, rel-avito user-visits / user-clicks).

**On Colab:** *Runtime → Change runtime type → T4 GPU*, then *Run all*. Results go to Drive (`authors_weights/results.csv`); the last cell downloads them. A task that runs out of memory is marked `failed` and the others continue.


In [ ]:
# ---- Settings ----
PROJECT_REPO = 'https://github.com/HARSHALK2598/cs224w-structural-optimizer'
PROJECT_BRANCH = 'step/authors-weights'
AUTHORS_REPO = 'https://github.com/cy623/Structural_Optimizer_RDL'
AUTHORS_COMMIT = '030b07ccf2b2eb43236a32e900e0f676448fe8f2'
WEIGHTS = ('https://github.com/cy623/Structural_Optimizer_RDL/releases/download/'
           'model_weight/structural_optimizer_rdl_weights.zip')
# Smallest first; each dataset is deleted after its tasks.
TASKS = [
    ('rel-f1', 'driver-top3'), ('rel-f1', 'driver-dnf'),
    ('rel-event', 'user-repeat'), ('rel-event', 'user-ignore'),
    ('rel-hm', 'user-churn'),
    ('rel-stack', 'user-engagement'), ('rel-stack', 'user-badge'),
    ('rel-amazon', 'item-churn'), ('rel-amazon', 'user-churn'),
]
USE_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/cs224w-structural-optimizer'


## 1. Code, environment, weights

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
HAS_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(
    ['nvidia-smi'], capture_output=True).returncode == 0
print('=' * 60 + f"\nRUNNING ON {'GPU' if HAS_GPU else 'CPU (no GPU found)'}\n" + '=' * 60, flush=True)


def sh(cmd, cwd=None, check=True):
    print('$', cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in proc.stdout:
        if 'it/s]' not in line and 'Warning' not in line:
            print(line, end='', flush=True)
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f'command failed ({code}): {cmd}')
    return code


if IN_COLAB:
    ROOT = Path('/content/cs224w-structural-optimizer')
    if not ROOT.exists():
        sh(f'git clone -q --branch {PROJECT_BRANCH} {PROJECT_REPO} {ROOT}')
else:
    ROOT = Path(os.environ.get('PROJECT_ROOT', Path.cwd())).resolve()
AUTHORS = ROOT / 'reference' / 'Structural_Optimizer_RDL'   # git-ignored
if not AUTHORS.exists():
    sh(f'git clone -q {AUTHORS_REPO} {AUTHORS}')
sh(f'git -C {AUTHORS} checkout -q {AUTHORS_COMMIT}')
sh(f'git -C {AUTHORS} log --oneline -1')

PY = AUTHORS / '.venv' / 'bin' / 'python'
if not PY.exists():
    if HAS_GPU:   # Their own installer (CUDA 12.6 wheels).
        sh('sh install.sh', cwd=AUTHORS)
    else:         # The same versions, CPU wheels.
        sh(f'{sys.executable} -m venv .venv', cwd=AUTHORS)
        pip = f'{PY} -m pip install -q'
        sh(f'{pip} torch==2.7.1 --index-url https://download.pytorch.org/whl/cpu', cwd=AUTHORS)
        sh(f'{pip} pyg-lib==0.4.0+pt27cpu torch-scatter==2.1.2+pt27cpu torch-sparse==0.6.18+pt27cpu '
           '--find-links https://data.pyg.org/whl/torch-2.7.0+cpu.html', cwd=AUTHORS)
        sh(f'{pip} -r requirements.txt', cwd=AUTHORS)
sh(f'{PY} -c "import torch, torch_geometric, torch_frame; print(torch.__version__, torch.cuda.is_available(), torch_geometric.__version__, torch_frame.__version__)"')

if not (AUTHORS / 'outputs' / 'best_model').exists():
    sh(f'curl -sSL -o /tmp/weights.zip {WEIGHTS}')
    import zipfile
    zipfile.ZipFile('/tmp/weights.zip').extractall(AUTHORS / 'outputs')
print(sorted(p.name for p in (AUTHORS / 'outputs' / 'best_model').iterdir()))


## 2. Where results go

In [ ]:
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path(DRIVE_DIR) / 'authors_weights'
else:
    OUT = ROOT / 'experiments' / 'results' / 'authors_weights'
OUT.mkdir(parents=True, exist_ok=True)
print('results:', OUT)
sh('free -g; df -h / | tail -1')


## 3. Prepare, predict, score

Each step is a fresh process of their `main.py`. `failed` means the step stopped (for example, out of memory).

In [ ]:
import json
gpu = '0' if HAS_GPU else 'cpu'
status = {}
previous = None
for dataset, task in TASKS:
    if previous and previous != dataset:  # Free the disk: their data and graphs.
        shutil.rmtree(AUTHORS / 'data' / 'raw' / previous, ignore_errors=True)
        for p in (AUTHORS / 'data' / 'graph').glob(previous + '__*'):
            shutil.rmtree(p, ignore_errors=True)
    previous = dataset
    print(f'\n======== {dataset} / {task} ========', flush=True)
    for mode in ['prepare', 'predict']:
        code = sh(f'{PY} main.py --mode {mode} --dataset {dataset} --task {task} --gpu {gpu}',
                  cwd=AUTHORS, check=False)
        if code != 0:
            status[f'{dataset}__{task}'] = f'failed at {mode} (exit {code})'
            break
    else:
        status[f'{dataset}__{task}'] = 'predicted'
        # Keep the task's labels (small) for scoring, before the data is deleted:
        labels = OUT / 'labels' / dataset / 'tasks' / task
        shutil.rmtree(labels, ignore_errors=True)
        shutil.copytree(AUTHORS / 'data' / 'raw' / dataset / 'tasks' / task, labels)
    sh('free -g | head -2', check=False)
print(json.dumps(status, indent=2))
(OUT / 'status.json').write_text(json.dumps(status, indent=2))


In [ ]:
# Our scoring: test ROC-AUC against RelBench's labels (the system Python is enough).
sh(f'{sys.executable} -m pip install -q pyyaml scikit-learn pandas pyarrow', check=False)
(OUT / 'predictions').mkdir(exist_ok=True)
for p in (AUTHORS / 'outputs' / 'predictions').glob('*.csv'):
    shutil.copy(p, OUT / 'predictions' / p.name)
sh(f'{sys.executable} experiments/benchmarks/score_authors_predictions.py '
   f'--predictions "{OUT / "predictions"}" --data_root "{OUT / "labels"}" '
   f'--out "{OUT / "results.csv"}"', cwd=ROOT, check=False)


## 4. Results

Test ROC-AUC (%) of the authors' released models, next to the paper's.

In [ ]:
import pandas as pd
res = pd.read_csv(OUT / 'results.csv')
print(res.to_string(index=False))
print('\nstatus:', json.dumps(status, indent=2))


In [ ]:
if IN_COLAB:
    from google.colab import files
    archive = shutil.make_archive('/content/authors_weights', 'zip', root_dir=OUT)
    files.download(archive)
